# 03 — Match the Dutch PEP Benchmark to OpenSanctions

This notebook evaluates OpenSanctions coverage against the independently constructed Dutch PEP benchmark snapshot exported by notebook 02.

The benchmark is treated as the reference dataset for the present analysis. OpenSanctions is the comparison dataset and is not assumed to be complete or error-free.

Matching is performed in stages:

1. exact matching on normalised names;
2. exact matching against OpenSanctions aliases;
3. candidate generation for unresolved records;
4. fuzzy matching for plausible remaining candidates;
5. manual review of uncertain matches.

A record without a confirmed match is described as not identified by the implemented matching procedure. It is not automatically interpreted as proof that the person is absent from OpenSanctions.

All results are specific to the current official-source benchmark snapshot and the dated OpenSanctions export used in this project.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import re
import unicodedata

# -------------------------------------------------------------------
# Project folder setup
# -------------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Main input files
# -------------------------------------------------------------------

BENCHMARK_PATH = (
    OUTPUT_DIR / "main_current_benchmark_v3_extended.csv"
)

OPENSANCTIONS_PATH = (
    EXTERNAL_DIR / "opensanctions_targets.simple.csv"
)

# -------------------------------------------------------------------
# Matching outputs
# -------------------------------------------------------------------

MATCH_RESULTS_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_match_results.csv"
)

MATCH_SUMMARY_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_match_summary.csv"
)

CATEGORY_COVERAGE_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_category_coverage.csv"
)

UNRESOLVED_CANDIDATES_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_unresolved_candidates.csv"
)

print("Project folder:", PROJECT_DIR)

print("\nInput files:")
print("Final benchmark exists:", BENCHMARK_PATH.exists())
print("OpenSanctions export exists:", OPENSANCTIONS_PATH.exists())

print("\nOutput paths:")
print("Match results:", MATCH_RESULTS_PATH)
print("Category coverage:", CATEGORY_COVERAGE_PATH)

Project folder: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean

Input files:
Final benchmark exists: True
OpenSanctions export exists: True

Output paths:
Match results: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_match_results.csv
Category coverage: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_category_coverage.csv


## 1. Load and inspect the benchmark and OpenSanctions data

Before matching, both files are inspected to confirm their schemas and to identify the fields available for names, aliases, country information, and OpenSanctions identifiers.

In [2]:
# -------------------------------------------------------------------
# Helper function for loading CSV files with common encodings
# -------------------------------------------------------------------

def read_csv_with_fallback_encodings(file_path):
    """
    Load a CSV file using common encodings.

    This supports both UTF-8 files and CSV files opened or saved
    through Excel using Windows-style encodings.
    """
    possible_encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in possible_encodings:
        try:
            dataframe = pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )

            print(f"Loaded {file_path.name} using: {encoding}")

            return dataframe, encoding

        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not load {file_path.name}. "
        f"Last error: {last_error}"
    )


# -------------------------------------------------------------------
# Load the benchmark and OpenSanctions files
# -------------------------------------------------------------------

benchmark_df, benchmark_encoding = (
    read_csv_with_fallback_encodings(BENCHMARK_PATH)
)

opensanctions_df, opensanctions_encoding = (
    read_csv_with_fallback_encodings(OPENSANCTIONS_PATH)
)

print("\nRow counts:")
print("Dutch PEP benchmark:", len(benchmark_df))
print("OpenSanctions records:", len(opensanctions_df))

print("\nBenchmark columns:")
print(benchmark_df.columns.tolist())

print("\nOpenSanctions columns:")
print(opensanctions_df.columns.tolist())

print("\nFirst three OpenSanctions records:")
display(opensanctions_df.head(3))

Loaded main_current_benchmark_v3_extended.csv using: utf-8
Loaded opensanctions_targets.simple.csv using: utf-8

Row counts:
Dutch PEP benchmark: 675
OpenSanctions records: 753980

Benchmark columns:
['benchmark_record_id', 'person_name', 'normalised_name', 'role_title', 'role_nl', 'role_eng', 'taxonomy_category', 'canonical_category', 'official_function', 'institutional_level', 'institution', 'party_name', 'source_url', 'page_title', 'extractor_used', 'access_timestamp_utc', 'validation_status', 'included_in_main_benchmark', 'operationalisation_status', 'source_row', 'category', 'source_component', 'evidence_text', 'review_notes', 'cleaning_notes', 'person_name_original', 'person_name_before_title_cleaning', 'benchmark_version']

OpenSanctions columns:
['id', 'schema', 'name', 'aliases', 'birth_date', 'countries', 'addresses', 'identifiers', 'sanctions', 'phones', 'emails', 'program_ids', 'dataset', 'first_seen', 'last_seen', 'last_change']

First three OpenSanctions records:


,id,schema,name,aliases,birth_date,countries,addresses,identifiers,sanctions,phones,emails,program_ids,dataset,first_seen,last_seen,last_change
0,NK-24KdLmG7SRFdS5GgJbgLSs,Person,"AW Kim Geok, PBS, PPA(P)",NaN,NaN,sg,NaN,NaN,NaN,NaN,aw_kim_geok@ite.edu.sg,NaN,Singapore Government Directory,2025-02-14T11:49:46,2026-06-03T00:29:05,2025-02-14T11:49:46
1,NK-24W9aosPLEsueQ5grGVNiK,Person,Christophane FOO,NaN,NaN,sg,NaN,NaN,NaN,NaN,Christophane_FOO@enterprisesg.gov.sg,NaN,Singapore Government Directory,2025-02-14T11:49:46,2026-06-03T00:29:05,2025-02-14T11:49:46
2,NK-24umfCYjCezXnS2gcCfT8F,Person,Andreas Norvik,NaN,1969-09-26,no,NaN,NaN,NaN,NaN,NaN,NaN,Norway State-Owned Enterprises Leadership,2026-02-23T22:56:15,2026-05-12T19:07:01,2026-02-23T22:56:15


## 2. Prepare OpenSanctions person names and aliases

OpenSanctions can store a person under a primary name and one or more aliases. This section creates one name-level matching table containing both.

Exact matching is performed against all OpenSanctions person records and aliases. This avoids incorrectly excluding Dutch benchmark members whose OpenSanctions country field is missing or refers to the country where they hold a diplomatic post.

A Dutch-relevant subset is also created for the later fuzzy-matching stage. Fuzzy matching is restricted to this subset to reduce false-positive candidates.

In [3]:
# -------------------------------------------------------------------
# Prepare OpenSanctions person names and aliases
# -------------------------------------------------------------------

import json

def normalise_for_matching(value):
    """
    Create a comparison-friendly name key.

    The function removes accents, punctuation, and repeated whitespace,
    while keeping the original name fields unchanged.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def create_clean_exact_key(value):
    """
    Create a second exact-match key for names with titles, degrees,
    or a first name provided in parentheses.

    Examples:
    - 'Mr. Th.C. (Thom) de Graaf' becomes 'thom de graaf'
    - 'Koning Willem-Alexander' becomes 'willem alexander'

    This does not infer full first names from initials.
    """
    if pd.isna(value):
        return ""

    raw_name = str(value).strip()

    parentheses_match = re.search(
        r"\(([^)]+)\)",
        raw_name
    )

    if parentheses_match:
        name_in_parentheses = (
            parentheses_match.group(1)
            .strip()
        )

        text_after_parentheses = (
            raw_name[parentheses_match.end():]
            .strip()
        )

        raw_name = (
            f"{name_in_parentheses} "
            f"{text_after_parentheses}"
        ).strip()

    normalised_name = normalise_for_matching(raw_name)

    title_and_degree_tokens = {
        "mr",
        "dr",
        "drs",
        "prof",
        "professor",
        "ir",
        "ing",
        "ma",
        "mba",
        "msc",
        "bsc",
        "ll",
        "llm",
        "koning",
        "king"
    }

    cleaned_tokens = [
        token
        for token in normalised_name.split()
        if token not in title_and_degree_tokens
    ]

    return " ".join(cleaned_tokens).strip()


def split_aliases(alias_value):
    """
    Convert an OpenSanctions alias field into a list of alias names.

    The function supports JSON-style lists and common separators such
    as semicolons and vertical bars.
    """
    if pd.isna(alias_value):
        return []

    alias_text = str(alias_value).strip()

    if alias_text == "":
        return []

    try:
        parsed_aliases = json.loads(alias_text)

        if isinstance(parsed_aliases, list):
            return [
                str(alias).strip()
                for alias in parsed_aliases
                if str(alias).strip()
            ]

        if isinstance(parsed_aliases, str):
            alias_text = parsed_aliases

    except (
        json.JSONDecodeError,
        TypeError
    ):
        pass

    return [
        alias.strip()
        for alias in re.split(
            r"[;|]",
            alias_text
        )
        if alias.strip()
    ]


# -------------------------------------------------------------------
# Keep only OpenSanctions person records
# -------------------------------------------------------------------

opensanctions_people_df = opensanctions_df[
    opensanctions_df["schema"]
    .astype("string")
    .eq("Person")
    .fillna(False)
].copy()

print("OpenSanctions person records:", len(opensanctions_people_df))


# -------------------------------------------------------------------
# Create primary-name records
# -------------------------------------------------------------------

os_primary_names_df = opensanctions_people_df[
    [
        "id",
        "name",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

os_primary_names_df = os_primary_names_df.rename(
    columns={
        "id": "opensanctions_id",
        "name": "opensanctions_name"
    }
)

os_primary_names_df["opensanctions_name_type"] = (
    "primary_name"
)


# -------------------------------------------------------------------
# Create alias-name records
# -------------------------------------------------------------------

os_alias_names_df = opensanctions_people_df[
    [
        "id",
        "aliases",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

os_alias_names_df = os_alias_names_df[
    os_alias_names_df["aliases"].notna()
].copy()

os_alias_names_df["opensanctions_name"] = (
    os_alias_names_df["aliases"]
    .apply(split_aliases)
)

os_alias_names_df = (
    os_alias_names_df
    .explode("opensanctions_name")
    .dropna(subset=["opensanctions_name"])
    .copy()
)

os_alias_names_df = os_alias_names_df[
    os_alias_names_df["opensanctions_name"]
    .astype("string")
    .str.strip()
    .ne("")
].copy()

os_alias_names_df = os_alias_names_df.rename(
    columns={
        "id": "opensanctions_id"
    }
)

os_alias_names_df["opensanctions_name_type"] = "alias"

os_alias_names_df = os_alias_names_df[
    [
        "opensanctions_id",
        "opensanctions_name",
        "opensanctions_name_type",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()


# -------------------------------------------------------------------
# Combine primary names and aliases into one matching table
# -------------------------------------------------------------------

os_names_all_df = pd.concat(
    [
        os_primary_names_df,
        os_alias_names_df
    ],
    ignore_index=True
)

os_names_all_df["strict_name_key"] = (
    os_names_all_df["opensanctions_name"]
    .apply(normalise_for_matching)
)

os_names_all_df["clean_exact_key"] = (
    os_names_all_df["opensanctions_name"]
    .apply(create_clean_exact_key)
)

os_names_all_df = os_names_all_df[
    os_names_all_df["strict_name_key"]
    .str.len()
    .gt(0)
].copy()

os_names_all_df["name_type_rank"] = (
    os_names_all_df["opensanctions_name_type"]
    .map(
        {
            "primary_name": 0,
            "alias": 1
        }
    )
    .fillna(9)
)

os_names_all_df = (
    os_names_all_df
    .drop_duplicates(
        subset=[
            "opensanctions_id",
            "opensanctions_name",
            "opensanctions_name_type"
        ]
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------
# Create a Dutch-relevant subset for later fuzzy matching
# -------------------------------------------------------------------

dutch_country_pattern = (
    r"(^|[^a-z])"
    r"(nl|nld|netherlands)"
    r"($|[^a-z])"
)

dutch_os_names_df = os_names_all_df[
    os_names_all_df["countries"]
    .fillna("")
    .astype(str)
    .str.contains(
        dutch_country_pattern,
        case=False,
        regex=True
    )
].copy()

print("\nOpenSanctions names:")
print("Primary-name rows:", len(os_primary_names_df))
print("Alias-name rows:", len(os_alias_names_df))
print("All names and aliases:", len(os_names_all_df))
print("Dutch-relevant names and aliases:", len(dutch_os_names_df))

print("\nExamples of alias records:")
display(
    os_names_all_df[
        os_names_all_df["opensanctions_name_type"]
        .eq("alias")
    ][
        [
            "opensanctions_id",
            "opensanctions_name",
            "countries",
            "dataset"
        ]
    ]
    .head(10)
)

OpenSanctions person records: 753980


C:\Users\steph\AppData\Local\Temp\ipykernel_8868\3286059621.py:321: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(



OpenSanctions names:
Primary-name rows: 753980
Alias-name rows: 139463
All names and aliases: 808004
Dutch-relevant names and aliases: 3103

Examples of alias records:


,opensanctions_id,opensanctions_name,countries,dataset
728479,NK-25bxtBgjBTs6asnevNVgsE,Shri Sambhaji Rao Shinde,in,India Lok and Rajya Sabha Members
728480,NK-2AaUhJdZeBxnxVKVF7fTVv,"Montemaggi, Marica",sm,MySociety's EveryPolitician Legislators;PEP po...
728481,NK-2MBHKUR939EFuVanhdHW4F,Shrimati Savitri Kulaste,in,India Lok and Rajya Sabha Members
728482,NK-2doDqFKpfJ3PdypTLj7mWP,Yeshwant Desai,in,India Lok and Rajya Sabha Members
728483,NK-2g9jbDDptYegUhxpHkcprg,"Vedrina Conesa, María Elisa",es,PEP position annotations by OpenSanctions;Spai...
728484,NK-2rqre8RgBxPpmrvDDVVTHN,Smt Munni Devi,in,India Lok and Rajya Sabha Members
728485,NK-4CP8FLdnNsCwMLiebBqDnr,Shrimati Damayantiraje Bhonsle,in,India Lok and Rajya Sabha Members
728486,NK-4EM3nAMNRA527CDstHAXvP,Smt. Sudama Devi,in,India Lok and Rajya Sabha Members
728487,NK-4QppmeHDSYVeESj7iHeCjH,Smt. Balwinder Kaur,in,India Lok and Rajya Sabha Members
728488,NK-4hjG5NRftefwmVgHCvpa7n,Smt. Madhavi,in,India Lok and Rajya Sabha Members


## 3. Exact matching against OpenSanctions names and aliases

Matching is performed in two exact stages:

1. strict normalised exact matching;
2. cleaned exact matching for unresolved records with titles, degrees, or first names shown in parentheses.

An exact match is counted only when the name key refers to one unique OpenSanctions person identifier. Exact-name collisions involving multiple OpenSanctions identifiers are retained as review cases rather than automatically counted as matches.

In [4]:
# -------------------------------------------------------------------
# Create one preferred OpenSanctions record per matching key
# -------------------------------------------------------------------

def create_key_summary(
    names_dataframe,
    key_column,
    output_prefix
):
    """
    Summarise OpenSanctions name records by one matching key.

    One preferred record is retained for display purposes, while the
    number of distinct OpenSanctions identifiers is retained to detect
    ambiguous exact-name collisions.
    """
    usable_names_df = names_dataframe[
        names_dataframe[key_column]
        .astype("string")
        .str.len()
        .gt(0)
    ].copy()

    usable_names_df = usable_names_df.sort_values(
        [
            key_column,
            "name_type_rank",
            "opensanctions_name"
        ]
    )

    preferred_name_df = (
        usable_names_df
        .drop_duplicates(
            subset=[key_column],
            keep="first"
        )
        [
            [
                key_column,
                "opensanctions_id",
                "opensanctions_name",
                "opensanctions_name_type",
                "countries",
                "dataset",
                "first_seen",
                "last_seen",
                "last_change"
            ]
        ]
        .copy()
    )

    preferred_name_df = preferred_name_df.rename(
        columns={
            "opensanctions_id": (
                f"{output_prefix}_opensanctions_id"
            ),
            "opensanctions_name": (
                f"{output_prefix}_opensanctions_name"
            ),
            "opensanctions_name_type": (
                f"{output_prefix}_opensanctions_name_type"
            ),
            "countries": f"{output_prefix}_countries",
            "dataset": f"{output_prefix}_dataset",
            "first_seen": f"{output_prefix}_first_seen",
            "last_seen": f"{output_prefix}_last_seen",
            "last_change": f"{output_prefix}_last_change"
        }
    )

    candidate_count_df = (
        usable_names_df
        .groupby(key_column)["opensanctions_id"]
        .nunique()
        .reset_index(
            name=f"{output_prefix}_candidate_id_count"
        )
    )

    return preferred_name_df.merge(
        candidate_count_df,
        on=key_column,
        how="left"
    )


# -------------------------------------------------------------------
# Create benchmark exact-match keys
# -------------------------------------------------------------------

benchmark_match_df = benchmark_df.copy()

benchmark_match_df["strict_name_key"] = (
    benchmark_match_df["person_name"]
    .apply(normalise_for_matching)
)

benchmark_match_df["clean_exact_key"] = (
    benchmark_match_df["person_name"]
    .apply(create_clean_exact_key)
)


# -------------------------------------------------------------------
# Strict exact matching
# -------------------------------------------------------------------

strict_key_summary_df = create_key_summary(
    names_dataframe=os_names_all_df,
    key_column="strict_name_key",
    output_prefix="strict"
)

exact_match_df = benchmark_match_df.merge(
    strict_key_summary_df,
    on="strict_name_key",
    how="left"
)

exact_match_df["strict_candidate_id_count"] = (
    exact_match_df["strict_candidate_id_count"]
    .fillna(0)
    .astype(int)
)

exact_match_df["strict_unique_match"] = (
    exact_match_df["strict_candidate_id_count"]
    .eq(1)
)


# -------------------------------------------------------------------
# Cleaned exact matching for records without a strict-name candidate
# -------------------------------------------------------------------

clean_key_summary_df = create_key_summary(
    names_dataframe=os_names_all_df,
    key_column="clean_exact_key",
    output_prefix="clean"
)

exact_match_df = exact_match_df.merge(
    clean_key_summary_df,
    on="clean_exact_key",
    how="left"
)

exact_match_df["clean_candidate_id_count"] = (
    exact_match_df["clean_candidate_id_count"]
    .fillna(0)
    .astype(int)
)

exact_match_df["clean_unique_match"] = (
    exact_match_df["strict_candidate_id_count"]
    .eq(0)
    &
    exact_match_df["clean_candidate_id_count"]
    .eq(1)
)


# -------------------------------------------------------------------
# Combine strict and cleaned exact results
# -------------------------------------------------------------------

exact_match_df["matched_in_exact_stage"] = False
exact_match_df["match_type"] = "no_exact_match"
exact_match_df["match_score"] = 0.0
exact_match_df["manual_review_needed"] = False

final_match_fields = [
    "opensanctions_id",
    "opensanctions_name",
    "opensanctions_name_type",
    "countries",
    "dataset",
    "first_seen",
    "last_seen",
    "last_change"
]

for field_name in final_match_fields:
    exact_match_df[field_name] = pd.NA


# Strict unique matches
strict_unique_mask = exact_match_df["strict_unique_match"]

exact_match_df.loc[
    strict_unique_mask,
    "matched_in_exact_stage"
] = True

exact_match_df.loc[
    strict_unique_mask,
    "match_type"
] = "normalised_exact"

exact_match_df.loc[
    strict_unique_mask,
    "match_score"
] = 100.0

for field_name in final_match_fields:
    exact_match_df.loc[
        strict_unique_mask,
        field_name
    ] = exact_match_df.loc[
        strict_unique_mask,
        f"strict_{field_name}"
    ]


# Cleaned unique matches
clean_unique_mask = exact_match_df["clean_unique_match"]

exact_match_df.loc[
    clean_unique_mask,
    "matched_in_exact_stage"
] = True

exact_match_df.loc[
    clean_unique_mask,
    "match_type"
] = "cleaned_exact"

exact_match_df.loc[
    clean_unique_mask,
    "match_score"
] = 100.0

for field_name in final_match_fields:
    exact_match_df.loc[
        clean_unique_mask,
        field_name
    ] = exact_match_df.loc[
        clean_unique_mask,
        f"clean_{field_name}"
    ]


# Exact-name collisions are not counted automatically.
strict_ambiguous_mask = (
    exact_match_df["strict_candidate_id_count"]
    .gt(1)
)

clean_ambiguous_mask = (
    exact_match_df["strict_candidate_id_count"]
    .eq(0)
    &
    exact_match_df["clean_candidate_id_count"]
    .gt(1)
)

exact_match_df.loc[
    strict_ambiguous_mask,
    "match_type"
] = "normalised_exact_ambiguous"

exact_match_df.loc[
    clean_ambiguous_mask,
    "match_type"
] = "cleaned_exact_ambiguous"

exact_match_df.loc[
    strict_ambiguous_mask
    | clean_ambiguous_mask,
    "manual_review_needed"
] = True


# -------------------------------------------------------------------
# Inspect exact matching results
# -------------------------------------------------------------------

print("Benchmark records:", len(exact_match_df))

print("\nExact matching results:")
display(
    exact_match_df["match_type"]
    .value_counts()
    .rename_axis("match_type")
    .reset_index(name="record_count")
)

print("\nConfirmed exact matches:")
print(
    int(
        exact_match_df[
            "matched_in_exact_stage"
        ].sum()
    )
)

print("\nExact-match records requiring review:")
print(
    int(
        exact_match_df[
            "manual_review_needed"
        ].sum()
    )
)

print("\nExact-match coverage by taxonomy category:")
display(
    exact_match_df
    .groupby("taxonomy_category")
    .agg(
        benchmark_records=(
            "benchmark_record_id",
            "size"
        ),
        exact_matches=(
            "matched_in_exact_stage",
            "sum"
        )
    )
    .assign(
        exact_coverage_rate=lambda dataframe: (
            dataframe["exact_matches"]
            / dataframe["benchmark_records"]
        )
    )
    .reset_index()
    .sort_values(
        "benchmark_records",
        ascending=False
    )
)

print("\nExamples of confirmed exact matches:")
display(
    exact_match_df[
        exact_match_df["matched_in_exact_stage"]
    ][
        [
            "benchmark_record_id",
            "person_name",
            "taxonomy_category",
            "opensanctions_name",
            "opensanctions_name_type",
            "opensanctions_id",
            "countries",
            "dataset",
            "match_type"
        ]
    ]
    .head(25)
)

Benchmark records: 675

Exact matching results:


,match_type,record_count
0,no_exact_match,393
1,normalised_exact,265
2,cleaned_exact,12
3,normalised_exact_ambiguous,5



Confirmed exact matches:
277

Exact-match records requiring review:
5

Exact-match coverage by taxonomy category:


,taxonomy_category,benchmark_records,exact_matches,exact_coverage_rate
9,house_members,150,146,0.973333
6,high_judiciary_crvb,86,0,0.000000
0,ambassadors,82,5,0.060976
11,registered_party_board,79,9,0.113924
12,senate_members,75,72,0.960000
8,high_judiciary_rvs,74,11,0.148649
5,high_judiciary_cbb,46,0,0.000000
7,high_judiciary_hr,34,0,0.000000
10,national_executive,28,28,1.000000
2,charges_d_affaires,7,0,0.000000



Examples of confirmed exact matches:


,benchmark_record_id,person_name,taxonomy_category,opensanctions_name,opensanctions_name_type,opensanctions_id,countries,dataset,match_type
9,10,Arjen van den Berg,ambassadors,Arjen van den Berg,primary_name,Q112661698,nl,Wikidata Persons in Relevant Categories,normalised_exact
13,14,Carin Lobbezoo,ambassadors,Carin Lobbezoo,primary_name,Q112661995,nl,Wikidata Persons in Relevant Categories,normalised_exact
57,58,Marjo Crompvoets,ambassadors,Marjo Crompvoets,primary_name,Q112661705,nl,Wikidata Persons in Relevant Categories,normalised_exact
62,63,Özlem Canel,ambassadors,Özlem Canel,primary_name,Q112661796,tr,Wikidata Persons in Relevant Categories,normalised_exact
74,75,Walter Oostelbos,ambassadors,Walter Oostelbos,primary_name,Q122870805,nl,PEP position annotations by OpenSanctions;Wiki...,normalised_exact
85,86,Olaf Sleijpen,central_bank_board,Olaf Sleijpen,primary_name,Q134962979,nl,PEP position annotations by OpenSanctions;US C...,normalised_exact
95,96,Ewout Irrgang,court_of_audit,Ewout Irrgang,primary_name,Q5419082,nl,Wikidata Persons in Relevant Categories,normalised_exact
96,97,Pieter Duisenberg,court_of_audit,Pieter Duisenberg,primary_name,Q1994265,nl;us,MySociety's EveryPolitician Legislators;PEP po...,normalised_exact
97,98,Koning Willem-Alexander,head_of_state,King Willem-Alexander,alias,Q154952,nl,PEP position annotations by OpenSanctions;UN H...,cleaned_exact
274,275,C.G. (Kees) van der Staaij,high_judiciary_rvs,Kees van der Staaij,primary_name,Q731640,nl,MySociety's EveryPolitician Legislators;PEP po...,cleaned_exact


## 4. Deterministic initial–surname matching

Some official Dutch sources represent first names as initials, whereas OpenSanctions may store full first names.

This stage identifies high-confidence matches when:

1. the benchmark record begins with one or more initials;
2. the full surname component matches exactly;
3. the first initial matches;
4. the rule identifies exactly one Dutch-relevant OpenSanctions person identifier.

These matches are recorded as deterministic rule-based matches and flagged for post-hoc audit. They are distinct from fuzzy candidate matches.

In [5]:
# -------------------------------------------------------------------
# Deterministic initial–surname unique matching
# -------------------------------------------------------------------
# This stage is intentionally stricter than fuzzy matching.
#
# It only accepts matches where:
# - the benchmark name starts with initials;
# - the complete surname component matches exactly;
# - the first initial matches;
# - exactly one OpenSanctions identifier satisfies the rule.
#
# Examples:
# P. Rosenmöller -> Paul Rosenmöller
# I.J. Visseren-Hamakers -> Ingrid J. Visseren-Hamakers
#
# It does not accept:
# Johanna Koffeman-Kramer -> Jan Kramer
# because "koffeman kramer" is not the same surname as "kramer".

INITIAL_SURNAME_MATCH_AUDIT_PATH = (
    OUTPUT_DIR
    / "benchmark_v3_opensanctions_initial_surname_matches.csv"
)


def extract_initial_surname_features(name_value):
    """
    Extract leading initials and the full remaining surname component.

    Examples:
    - 'P. Rosenmöller'
      -> initials ['p'], surname 'rosenmoller'

    - 'I.J. Visseren-Hamakers'
      -> initials ['i', 'j'], surname 'visseren hamakers'

    - 'Paul Rosenmöller'
      -> no initials, blank surname result for this matching stage.
    """
    name_key = normalise_for_matching(name_value)

    if name_key == "":
        return {
            "leading_initials": [],
            "surname_key": "",
            "first_initial": ""
        }

    tokens = name_key.split()

    leading_initials = []
    token_position = 0

    while (
        token_position < len(tokens)
        and re.fullmatch(
            r"[a-z]",
            tokens[token_position]
        )
    ):
        leading_initials.append(
            tokens[token_position]
        )

        token_position += 1

    surname_tokens = tokens[token_position:]

    # This matching stage applies only to names that begin with initials.
    if len(leading_initials) == 0 or len(surname_tokens) == 0:
        return {
            "leading_initials": [],
            "surname_key": "",
            "first_initial": ""
        }

    return {
        "leading_initials": leading_initials,
        "surname_key": " ".join(surname_tokens),
        "first_initial": leading_initials[0]
    }


def candidate_ends_with_full_surname(
    candidate_name_key,
    benchmark_surname_key
):
    """
    Check whether an OpenSanctions name ends with the benchmark's full
    surname component.

    This is stricter than comparing only the final surname token.
    """
    if (
        candidate_name_key == ""
        or benchmark_surname_key == ""
    ):
        return False

    return (
        candidate_name_key == benchmark_surname_key
        or candidate_name_key.endswith(
            f" {benchmark_surname_key}"
        )
    )


def candidate_first_initial(candidate_name_key):
    """
    Return the first letter of the candidate's first token.
    """
    if pd.isna(candidate_name_key):
        return ""

    candidate_tokens = str(candidate_name_key).split()

    if len(candidate_tokens) == 0:
        return ""

    return candidate_tokens[0][0]


# -------------------------------------------------------------------
# Select unresolved records after exact matching
# -------------------------------------------------------------------

initial_surname_unresolved_df = exact_match_df[
    exact_match_df["match_type"]
    .eq("no_exact_match")
].copy()

initial_features_df = (
    initial_surname_unresolved_df["person_name"]
    .apply(extract_initial_surname_features)
    .apply(pd.Series)
)

initial_surname_unresolved_df = pd.concat(
    [
        initial_surname_unresolved_df.reset_index(drop=True),
        initial_features_df.reset_index(drop=True)
    ],
    axis=1
)

initial_surname_unresolved_df = (
    initial_surname_unresolved_df[
        initial_surname_unresolved_df["surname_key"]
        .astype("string")
        .str.len()
        .gt(0)
    ]
    .copy()
)

print(
    "Unresolved benchmark records with leading initials:",
    len(initial_surname_unresolved_df)
)


# -------------------------------------------------------------------
# Prepare Dutch-relevant OpenSanctions candidates
# -------------------------------------------------------------------

initial_surname_candidate_pool_df = (
    dutch_os_names_df.copy()
)

initial_surname_candidate_pool_df = (
    initial_surname_candidate_pool_df[
        initial_surname_candidate_pool_df["clean_exact_key"]
        .astype("string")
        .str.len()
        .gt(0)
    ]
    .copy()
)

initial_surname_candidate_pool_df[
    "candidate_first_initial"
] = (
    initial_surname_candidate_pool_df[
        "clean_exact_key"
    ]
    .apply(candidate_first_initial)
)


# -------------------------------------------------------------------
# Find unique initial–surname candidates
# -------------------------------------------------------------------

initial_surname_match_rows = []

for _, benchmark_row in initial_surname_unresolved_df.iterrows():

    benchmark_surname_key = benchmark_row["surname_key"]
    benchmark_first_initial = benchmark_row["first_initial"]

    possible_candidates_df = (
        initial_surname_candidate_pool_df[
            initial_surname_candidate_pool_df[
                "candidate_first_initial"
            ]
            .eq(benchmark_first_initial)
        ]
        .copy()
    )

    possible_candidates_df = possible_candidates_df[
        possible_candidates_df[
            "clean_exact_key"
        ]
        .apply(
            lambda candidate_key: (
                candidate_ends_with_full_surname(
                    candidate_name_key=candidate_key,
                    benchmark_surname_key=benchmark_surname_key
                )
            )
        )
    ].copy()

    unique_candidate_ids = (
        possible_candidates_df[
            "opensanctions_id"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    if len(unique_candidate_ids) != 1:
        continue

    selected_candidate_id = unique_candidate_ids[0]

    selected_candidate_df = (
        possible_candidates_df[
            possible_candidates_df[
                "opensanctions_id"
            ]
            .eq(selected_candidate_id)
        ]
        .sort_values(
            [
                "name_type_rank",
                "opensanctions_name"
            ]
        )
        .head(1)
    )

    selected_candidate = selected_candidate_df.iloc[0]

    initial_surname_match_rows.append({
        "benchmark_record_id": benchmark_row[
            "benchmark_record_id"
        ],
        "person_name": benchmark_row["person_name"],
        "taxonomy_category": benchmark_row[
            "taxonomy_category"
        ],
        "role_title": benchmark_row["role_title"],
        "institution": benchmark_row["institution"],
        "benchmark_surname_key": benchmark_surname_key,
        "benchmark_first_initial": benchmark_first_initial,
        "opensanctions_id": selected_candidate[
            "opensanctions_id"
        ],
        "opensanctions_name": selected_candidate[
            "opensanctions_name"
        ],
        "opensanctions_name_type": selected_candidate[
            "opensanctions_name_type"
        ],
        "countries": selected_candidate["countries"],
        "dataset": selected_candidate["dataset"],
        "first_seen": selected_candidate["first_seen"],
        "last_seen": selected_candidate["last_seen"],
        "last_change": selected_candidate["last_change"],
        "match_type": "initial_surname_unique",
        "match_score": 100.0,
        "match_confidence": "high",
        "review_status": (
            "rule_based_match_flagged_for_post_hoc_audit"
        ),
        "rule_description": (
            "Leading initial, complete surname component, and one "
            "unique Dutch-relevant OpenSanctions identifier matched."
        )
    })


initial_surname_matches_df = pd.DataFrame(
    initial_surname_match_rows
)

initial_surname_matches_df.to_csv(
    INITIAL_SURNAME_MATCH_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nConfirmed initial–surname unique matches:",
    len(initial_surname_matches_df)
)

print("\nSaved rule-based match audit:")
print(INITIAL_SURNAME_MATCH_AUDIT_PATH)

display(
    initial_surname_matches_df[
        [
            "benchmark_record_id",
            "person_name",
            "taxonomy_category",
            "opensanctions_name",
            "opensanctions_id",
            "dataset",
            "match_type",
            "review_status"
        ]
    ]
    .sort_values(
        [
            "taxonomy_category",
            "person_name"
        ]
    )
)

Unresolved benchmark records with leading initials: 231

Confirmed initial–surname unique matches: 8

Saved rule-based match audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_initial_surname_matches.csv


,benchmark_record_id,person_name,taxonomy_category,opensanctions_name,opensanctions_id,dataset,match_type,review_status
0,202,M.M. den Boer,high_judiciary_crvb,Monica den Boer,Q2270440,MySociety's EveryPolitician Legislators;PEP po...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
1,241,F.G.F. Peters,high_judiciary_hr,Frederik Peters,Q106119790,Wikidata Persons in Relevant Categories;Wikida...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
2,246,G. de Groot,high_judiciary_hr,Gert de Groot,Q5865178,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
3,255,M.V. Polak,high_judiciary_hr,Marvin Polak,Q106204204,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
4,256,M.W.C. Feteris,high_judiciary_hr,Maarten Feteris,Q17428797,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
5,258,R. Kuiper,high_judiciary_hr,Roel Kuiper,Q3057862,Wikidata Persons in Relevant Categories,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
6,262,T. Kooijmans,high_judiciary_hr,Tiny Kooijmans,Q3931118,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit
7,654,R.S. Croll,senate_members,Robert Croll,Q117193203,Netherlands Senate;Wikidata Persons in Relevan...,initial_surname_unique,rule_based_match_flagged_for_post_hoc_audit


In [6]:
# -------------------------------------------------------------------
# Add deterministic initial–surname matches to the matching dataset
# -------------------------------------------------------------------

if len(initial_surname_matches_df) > 0:

    initial_surname_merge_columns = [
        "benchmark_record_id",
        "opensanctions_id",
        "opensanctions_name",
        "opensanctions_name_type",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change",
        "match_type",
        "match_score",
        "match_confidence",
        "review_status"
    ]

    exact_match_df = exact_match_df.merge(
        initial_surname_matches_df[
            initial_surname_merge_columns
        ].rename(
            columns={
                "opensanctions_id": (
                    "rule_opensanctions_id"
                ),
                "opensanctions_name": (
                    "rule_opensanctions_name"
                ),
                "opensanctions_name_type": (
                    "rule_opensanctions_name_type"
                ),
                "countries": "rule_countries",
                "dataset": "rule_dataset",
                "first_seen": "rule_first_seen",
                "last_seen": "rule_last_seen",
                "last_change": "rule_last_change",
                "match_type": "rule_match_type",
                "match_score": "rule_match_score",
                "match_confidence": (
                    "rule_match_confidence"
                ),
                "review_status": "rule_review_status"
            }
        ),
        on="benchmark_record_id",
        how="left"
    )

    initial_surname_match_mask = (
        exact_match_df["rule_match_type"]
        .astype("string")
        .eq("initial_surname_unique")
        .fillna(False)
    )

    exact_match_df.loc[
        initial_surname_match_mask,
        "matched_in_exact_stage"
    ] = True

    exact_match_df.loc[
        initial_surname_match_mask,
        "match_type"
    ] = "initial_surname_unique"

    exact_match_df.loc[
        initial_surname_match_mask,
        "match_score"
    ] = 100.0

    for final_field, rule_field in {
        "opensanctions_id": "rule_opensanctions_id",
        "opensanctions_name": "rule_opensanctions_name",
        "opensanctions_name_type": (
            "rule_opensanctions_name_type"
        ),
        "countries": "rule_countries",
        "dataset": "rule_dataset",
        "first_seen": "rule_first_seen",
        "last_seen": "rule_last_seen",
        "last_change": "rule_last_change"
    }.items():
        exact_match_df.loc[
            initial_surname_match_mask,
            final_field
        ] = exact_match_df.loc[
            initial_surname_match_mask,
            rule_field
        ]

    exact_match_df.loc[
        initial_surname_match_mask,
        "manual_review_needed"
    ] = True

else:
    exact_match_df["match_confidence"] = pd.NA
    exact_match_df["review_status"] = pd.NA

# Add transparent match-confidence fields for every stage.
if "match_confidence" not in exact_match_df.columns:
    exact_match_df["match_confidence"] = pd.NA

if "review_status" not in exact_match_df.columns:
    exact_match_df["review_status"] = pd.NA

exact_match_df.loc[
    exact_match_df["match_type"].isin(
        [
            "normalised_exact",
            "cleaned_exact"
        ]
    ),
    "match_confidence"
] = "high"

exact_match_df.loc[
    exact_match_df["match_type"].isin(
        [
            "normalised_exact",
            "cleaned_exact"
        ]
    ),
    "review_status"
] = "automatic_exact_match"

exact_match_df.loc[
    exact_match_df["match_type"]
    .eq("initial_surname_unique"),
    "match_confidence"
] = "high"

exact_match_df.loc[
    exact_match_df["match_type"]
    .eq("initial_surname_unique"),
    "review_status"
] = "rule_based_match_flagged_for_post_hoc_audit"

print("Matching results after deterministic initial–surname stage:")

display(
    exact_match_df[
        "match_type"
    ]
    .value_counts()
    .rename_axis("match_type")
    .reset_index(name="record_count")
)

print(
    "\nConfirmed matches after deterministic matching:",
    int(
        exact_match_df[
            "matched_in_exact_stage"
        ].sum()
    )
)

print(
    "\nRule-based matches flagged for later audit:",
    int(
        exact_match_df[
            "match_type"
        ]
        .eq("initial_surname_unique")
        .sum()
    )
)

Matching results after deterministic initial–surname stage:


,match_type,record_count
0,no_exact_match,385
1,normalised_exact,265
2,cleaned_exact,12
3,initial_surname_unique,8
4,normalised_exact_ambiguous,5



Confirmed matches after deterministic matching: 285

Rule-based matches flagged for later audit: 8


## 5. Fuzzy candidate generation for unresolved benchmark records

Fuzzy matching is used only to generate and prioritise candidate OpenSanctions records for manual review.

It is not treated as automatic evidence of identity. A candidate is only counted as a confirmed match after manual verification of the person, role context, and OpenSanctions record.

Candidate generation is limited to Dutch-relevant OpenSanctions names and aliases. This reduces the risk of matching Dutch benchmark members to similarly named people in other countries.

In [7]:
# -------------------------------------------------------------------
# Create fuzzy candidate matches for unresolved benchmark records
# -------------------------------------------------------------------
# RapidFuzz is used for fast string comparison. The results below are
# candidate matches only and are not counted as confirmed coverage.

from rapidfuzz import fuzz, process

FUZZY_CANDIDATES_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_fuzzy_candidates.csv"
)

FUZZY_REVIEW_QUEUE_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_fuzzy_review_queue.csv"
)


def get_last_name_token(name_key):
    """
    Return the final token of a normalised name.

    This is used only as an additional candidate-quality signal.
    """
    if pd.isna(name_key) or str(name_key).strip() == "":
        return ""

    return str(name_key).strip().split()[-1]


def get_first_initial(name_key):
    """
    Return the first alphabetic character in a normalised name.
    """
    if pd.isna(name_key):
        return ""

    letters_only = re.sub(
        r"[^a-z]",
        "",
        str(name_key).lower()
    )

    return letters_only[0] if letters_only else ""


def classify_fuzzy_candidate(
    score,
    same_surname,
    first_initial_matches
):
    """
    Assign a review priority without automatically accepting a match.

    High-priority candidates have both a strong similarity score and
    surname agreement. All candidates still require manual review.
    """
    if (
        score >= 90
        and same_surname
    ):
        return "high_priority_review"

    if (
        score >= 80
        and same_surname
        and first_initial_matches
    ):
        return "medium_priority_review"

    if (
        score >= 75
        and same_surname
    ):
        return "low_priority_review"

    return "no_reliable_candidate"


# -------------------------------------------------------------------
# Select records without an exact match
# -------------------------------------------------------------------

unresolved_for_fuzzy_df = exact_match_df[
    exact_match_df["match_type"]
    .eq("no_exact_match")
].copy()

unresolved_for_fuzzy_df["fuzzy_benchmark_key"] = (
    unresolved_for_fuzzy_df["person_name"]
    .apply(create_clean_exact_key)
)

unresolved_for_fuzzy_df["benchmark_last_name"] = (
    unresolved_for_fuzzy_df["fuzzy_benchmark_key"]
    .apply(get_last_name_token)
)

unresolved_for_fuzzy_df["benchmark_first_initial"] = (
    unresolved_for_fuzzy_df["fuzzy_benchmark_key"]
    .apply(get_first_initial)
)

print("Records entering fuzzy candidate generation:")
print(len(unresolved_for_fuzzy_df))


# -------------------------------------------------------------------
# Prepare Dutch-relevant OpenSanctions names and aliases
# -------------------------------------------------------------------

fuzzy_candidate_pool_df = dutch_os_names_df.copy()

fuzzy_candidate_pool_df = fuzzy_candidate_pool_df[
    fuzzy_candidate_pool_df["clean_exact_key"]
    .astype("string")
    .str.len()
    .gt(0)
].copy()

fuzzy_candidate_pool_df["candidate_last_name"] = (
    fuzzy_candidate_pool_df["clean_exact_key"]
    .apply(get_last_name_token)
)

fuzzy_candidate_pool_df["candidate_first_initial"] = (
    fuzzy_candidate_pool_df["clean_exact_key"]
    .apply(get_first_initial)
)

fuzzy_candidate_keys = (
    fuzzy_candidate_pool_df["clean_exact_key"]
    .tolist()
)

print("Dutch-relevant OpenSanctions candidate names:")
print(len(fuzzy_candidate_pool_df))


# -------------------------------------------------------------------
# Retrieve the top candidates for each unresolved benchmark record
# -------------------------------------------------------------------
# We retrieve more than three raw results because one OpenSanctions
# person can appear under both a primary name and multiple aliases.

fuzzy_candidate_rows = []

for _, benchmark_row in unresolved_for_fuzzy_df.iterrows():

    benchmark_key = benchmark_row["fuzzy_benchmark_key"]

    if benchmark_key == "":
        continue

    raw_matches = process.extract(
        benchmark_key,
        fuzzy_candidate_keys,
        scorer=fuzz.WRatio,
        limit=20
    )

    seen_opensanctions_ids = set()
    candidate_rank = 0

    for _, score, candidate_position in raw_matches:

        candidate_row = fuzzy_candidate_pool_df.iloc[
            candidate_position
        ]

        candidate_id = candidate_row["opensanctions_id"]

        # Keep one best name representation per OpenSanctions person.
        if candidate_id in seen_opensanctions_ids:
            continue

        seen_opensanctions_ids.add(candidate_id)

        same_surname = (
            benchmark_row["benchmark_last_name"]
            == candidate_row["candidate_last_name"]
        )

        first_initial_matches = (
            benchmark_row["benchmark_first_initial"]
            == candidate_row["candidate_first_initial"]
        )

        priority = classify_fuzzy_candidate(
            score=round(float(score), 2),
            same_surname=same_surname,
            first_initial_matches=first_initial_matches
        )

        candidate_rank += 1

        fuzzy_candidate_rows.append({
            "benchmark_record_id": benchmark_row[
                "benchmark_record_id"
            ],
            "person_name": benchmark_row["person_name"],
            "taxonomy_category": benchmark_row[
                "taxonomy_category"
            ],
            "role_title": benchmark_row["role_title"],
            "institution": benchmark_row["institution"],
            "fuzzy_benchmark_key": benchmark_key,
            "benchmark_last_name": benchmark_row[
                "benchmark_last_name"
            ],
            "candidate_rank": candidate_rank,
            "candidate_score": round(float(score), 2),
            "same_surname": same_surname,
            "first_initial_matches": first_initial_matches,
            "candidate_priority": priority,
            "candidate_opensanctions_id": candidate_id,
            "candidate_opensanctions_name": candidate_row[
                "opensanctions_name"
            ],
            "candidate_opensanctions_name_type": candidate_row[
                "opensanctions_name_type"
            ],
            "candidate_countries": candidate_row["countries"],
            "candidate_dataset": candidate_row["dataset"],
            "candidate_first_seen": candidate_row["first_seen"],
            "candidate_last_seen": candidate_row["last_seen"],
            "review_decision": "needs_manual_review",
            "review_notes": pd.NA
        })

        # Retain at most three distinct OpenSanctions people per record.
        if candidate_rank == 3:
            break


fuzzy_candidates_df = pd.DataFrame(
    fuzzy_candidate_rows
)

# Save all candidate rows for auditability.
fuzzy_candidates_df.to_csv(
    FUZZY_CANDIDATES_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Create a practical review queue. Low-confidence candidates remain
# saved in the full audit file but do not need immediate review.
fuzzy_review_queue_df = fuzzy_candidates_df[
    fuzzy_candidates_df["candidate_priority"]
    .isin(
        [
            "high_priority_review",
            "medium_priority_review"
        ]
    )
].copy()

fuzzy_review_queue_df.to_csv(
    FUZZY_REVIEW_QUEUE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nFuzzy candidate rows created:")
print(len(fuzzy_candidates_df))

print("\nUnique unresolved benchmark records with candidates:")
print(
    fuzzy_candidates_df[
        "benchmark_record_id"
    ].nunique()
)

print("\nCandidate priority counts:")
display(
    fuzzy_candidates_df[
        "candidate_priority"
    ]
    .value_counts()
    .rename_axis("candidate_priority")
    .reset_index(name="candidate_rows")
)

print("\nUnique benchmark records in the review queue:")
print(
    fuzzy_review_queue_df[
        "benchmark_record_id"
    ].nunique()
)

print("\nSaved full fuzzy candidate audit:")
print(FUZZY_CANDIDATES_PATH)

print("\nSaved prioritised fuzzy review queue:")
print(FUZZY_REVIEW_QUEUE_PATH)

print("\nHighest-priority candidates:")
display(
    fuzzy_review_queue_df[
        [
            "benchmark_record_id",
            "person_name",
            "taxonomy_category",
            "role_title",
            "candidate_score",
            "same_surname",
            "first_initial_matches",
            "candidate_opensanctions_name",
            "candidate_opensanctions_id",
            "candidate_dataset",
            "candidate_priority"
        ]
    ]
    .sort_values(
        [
            "candidate_priority",
            "candidate_score"
        ],
        ascending=[True, False]
    )
    .head(50)
)

Records entering fuzzy candidate generation:
385
Dutch-relevant OpenSanctions candidate names:
3103

Fuzzy candidate rows created:
1155

Unique unresolved benchmark records with candidates:
385

Candidate priority counts:


,candidate_priority,candidate_rows
0,no_reliable_candidate,1097
1,low_priority_review,51
2,medium_priority_review,4
3,high_priority_review,3



Unique benchmark records in the review queue:
6

Saved full fuzzy candidate audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_fuzzy_candidates.csv

Saved prioritised fuzzy review queue:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_fuzzy_review_queue.csv

Highest-priority candidates:


,benchmark_record_id,person_name,taxonomy_category,role_title,candidate_score,same_surname,first_initial_matches,candidate_opensanctions_name,candidate_opensanctions_id,candidate_dataset,candidate_priority
87,32,Henk Jan Bakker,ambassadors,Ambassadeur in Kenia,95.0,True,True,Henk Bakker,Q2279573,PEP position annotations by OpenSanctions;Wiki...,high_priority_review
1023,555,Klaas-Jan de Ruiter,registered_party_board,Vice-voorzitter,95.0,True,False,Jan de Ruiter,Q3074664,PEP position annotations by OpenSanctions;Wiki...,high_priority_review
1143,645,Meryem Karaaslan,senate_members,Lid van de Eerste Kamer der Staten-Generaal,95.0,True,True,Meryem Kılıç-Karaaslan,Q117235809,Netherlands Senate;Wikidata Persons in Relevan...,high_priority_review
22,8,Anneke Luwema,ambassadors,Ambassadeur in Algerije,85.5,True,True,Anneke Elisabeth Luwema,Q112661550,Wikidata Persons in Relevant Categories;Wikida...,medium_priority_review
1008,548,Johanna Koffeman-Kramer,registered_party_board,Internationaal Secretaris Landelijk Bestuur,85.5,True,True,Jan Kramer,Q130595704,Wikidata Persons in Relevant Categories;Wikida...,medium_priority_review
1010,548,Johanna Koffeman-Kramer,registered_party_board,Internationaal Secretaris Landelijk Bestuur,85.5,True,True,Johannes Kramer,Q2410039,PEP position annotations by OpenSanctions;Wiki...,medium_priority_review
1148,671,André Steur,senior_military,Commandant Luchtstrijdkrachten,85.5,True,True,Ard van der Steur,Q4787767,MySociety's EveryPolitician Legislators;PEP po...,medium_priority_review


## 6. Final match classification and coverage measures

The final analysis distinguishes between two coverage measures:

1. **Strict confirmed coverage**: normalised exact and cleaned exact matches only.
2. **Extended rule-based coverage**: strict confirmed matches plus unique initial–surname matches.

Initial–surname matches are retained as rule-based probable matches and flagged for post-hoc audit. Fuzzy candidates are not counted as matches because fuzzy similarity alone does not provide sufficient evidence of identity.

This separation provides a transparent sensitivity analysis: strict coverage is the conservative estimate, while extended coverage reflects identifiable abbreviated-name variants in official Dutch sources.

In [8]:
# -------------------------------------------------------------------
# Finalise match classifications and calculate coverage measures
# -------------------------------------------------------------------
# Strict coverage includes only exact matching stages.
# Extended coverage adds unique initial-surname matches.
# Fuzzy candidates remain review candidates and are not counted.

def write_csv_safely(dataframe, output_path):
    """
    Save a CSV file with UTF-8 encoding.

    If the intended file is open in Excel, a timestamped fallback
    file is created rather than stopping the notebook.
    """
    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    except PermissionError:
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%d_%H%M%S")

        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )

        dataframe.to_csv(
            fallback_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Warning: output file is open. "
            f"Saved fallback file: {fallback_path.name}"
        )

        return fallback_path


# -------------------------------------------------------------------
# Define transparent match-status fields
# -------------------------------------------------------------------

final_match_results_df = exact_match_df.copy()

strict_exact_match_types = [
    "normalised_exact",
    "cleaned_exact"
]

rule_based_match_types = [
    "initial_surname_unique"
]

final_match_results_df["strict_confirmed_match"] = (
    final_match_results_df["match_type"]
    .isin(strict_exact_match_types)
)

final_match_results_df["rule_based_probable_match"] = (
    final_match_results_df["match_type"]
    .isin(rule_based_match_types)
)

final_match_results_df["included_in_extended_coverage"] = (
    final_match_results_df["strict_confirmed_match"]
    | final_match_results_df["rule_based_probable_match"]
)

final_match_results_df["final_match_status"] = (
    "not_identified_by_implemented_matching"
)

final_match_results_df.loc[
    final_match_results_df["strict_confirmed_match"],
    "final_match_status"
] = "confirmed_automatic_match"

final_match_results_df.loc[
    final_match_results_df["rule_based_probable_match"],
    "final_match_status"
] = "probable_rule_based_match"

final_match_results_df.loc[
    final_match_results_df["match_type"]
    .isin(
        [
            "normalised_exact_ambiguous",
            "cleaned_exact_ambiguous"
        ]
    ),
    "final_match_status"
] = "ambiguous_exact_name_candidate"

# -------------------------------------------------------------------
# Add the best fuzzy review candidate for unresolved records
# -------------------------------------------------------------------

best_fuzzy_candidate_df = (
    fuzzy_candidates_df
    .sort_values(
        [
            "benchmark_record_id",
            "candidate_score",
            "candidate_rank"
        ],
        ascending=[True, False, True]
    )
    .drop_duplicates(
        subset=["benchmark_record_id"],
        keep="first"
    )
    [
        [
            "benchmark_record_id",
            "candidate_score",
            "candidate_priority",
            "candidate_opensanctions_id",
            "candidate_opensanctions_name",
            "candidate_dataset"
        ]
    ]
    .rename(
        columns={
            "candidate_score": "best_fuzzy_score",
            "candidate_priority": "best_fuzzy_priority",
            "candidate_opensanctions_id": (
                "best_fuzzy_opensanctions_id"
            ),
            "candidate_opensanctions_name": (
                "best_fuzzy_opensanctions_name"
            ),
            "candidate_dataset": "best_fuzzy_dataset"
        }
    )
)

final_match_results_df = final_match_results_df.merge(
    best_fuzzy_candidate_df,
    on="benchmark_record_id",
    how="left"
)

# -------------------------------------------------------------------
# Create match-summary output
# -------------------------------------------------------------------

benchmark_total = len(final_match_results_df)

strict_match_count = int(
    final_match_results_df[
        "strict_confirmed_match"
    ].sum()
)

rule_based_match_count = int(
    final_match_results_df[
        "rule_based_probable_match"
    ].sum()
)

extended_match_count = int(
    final_match_results_df[
        "included_in_extended_coverage"
    ].sum()
)

ambiguous_exact_count = int(
    final_match_results_df[
        "final_match_status"
    ]
    .eq("ambiguous_exact_name_candidate")
    .sum()
)

unresolved_count = int(
    final_match_results_df[
        "final_match_status"
    ]
    .eq("not_identified_by_implemented_matching")
    .sum()
)

fuzzy_review_record_count = int(
    fuzzy_review_queue_df[
        "benchmark_record_id"
    ].nunique()
)

match_summary_df = pd.DataFrame([
    {
        "measure": "benchmark_records",
        "record_count": benchmark_total,
        "coverage_rate": 1.0,
        "definition": (
            "All records in the final Dutch official-source benchmark."
        )
    },
    {
        "measure": "strict_confirmed_matches",
        "record_count": strict_match_count,
        "coverage_rate": strict_match_count / benchmark_total,
        "definition": (
            "Unique normalised exact and cleaned exact matches."
        )
    },
    {
        "measure": "rule_based_initial_surname_matches",
        "record_count": rule_based_match_count,
        "coverage_rate": rule_based_match_count / benchmark_total,
        "definition": (
            "Unique Dutch-relevant OpenSanctions IDs with matching "
            "full surname and leading first initial; flagged for audit."
        )
    },
    {
        "measure": "extended_rule_based_matches",
        "record_count": extended_match_count,
        "coverage_rate": extended_match_count / benchmark_total,
        "definition": (
            "Strict confirmed matches plus rule-based initial-surname "
            "matches."
        )
    },
    {
        "measure": "ambiguous_exact_name_candidates",
        "record_count": ambiguous_exact_count,
        "coverage_rate": ambiguous_exact_count / benchmark_total,
        "definition": (
            "Exact-name collisions involving more than one "
            "OpenSanctions identifier; not counted as matches."
        )
    },
    {
        "measure": "unresolved_after_deterministic_matching",
        "record_count": unresolved_count,
        "coverage_rate": unresolved_count / benchmark_total,
        "definition": (
            "Records not identified by the implemented deterministic "
            "matching procedure."
        )
    },
    {
        "measure": "unresolved_records_with_fuzzy_review_candidates",
        "record_count": fuzzy_review_record_count,
        "coverage_rate": fuzzy_review_record_count / benchmark_total,
        "definition": (
            "Unresolved records with high- or medium-priority fuzzy "
            "candidates; not counted as matches."
        )
    }
])

# -------------------------------------------------------------------
# Calculate coverage by PEP taxonomy category
# -------------------------------------------------------------------

category_coverage_df = (
    final_match_results_df
    .groupby("taxonomy_category")
    .agg(
        benchmark_records=(
            "benchmark_record_id",
            "size"
        ),
        strict_confirmed_matches=(
            "strict_confirmed_match",
            "sum"
        ),
        rule_based_initial_surname_matches=(
            "rule_based_probable_match",
            "sum"
        ),
        extended_rule_based_matches=(
            "included_in_extended_coverage",
            "sum"
        ),
        ambiguous_exact_candidates=(
            "final_match_status",
            lambda values: (
                values
                .eq("ambiguous_exact_name_candidate")
                .sum()
            )
        ),
        unresolved_after_deterministic_matching=(
            "final_match_status",
            lambda values: (
                values
                .eq(
                    "not_identified_by_implemented_matching"
                )
                .sum()
            )
        )
    )
    .reset_index()
)

category_coverage_df["strict_coverage_rate"] = (
    category_coverage_df["strict_confirmed_matches"]
    / category_coverage_df["benchmark_records"]
)

category_coverage_df["extended_coverage_rate"] = (
    category_coverage_df["extended_rule_based_matches"]
    / category_coverage_df["benchmark_records"]
)

category_coverage_df = category_coverage_df.sort_values(
    "benchmark_records",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------------------------
# Save final outputs
# -------------------------------------------------------------------

match_results_output_path = write_csv_safely(
    final_match_results_df,
    MATCH_RESULTS_PATH
)

match_summary_output_path = write_csv_safely(
    match_summary_df,
    MATCH_SUMMARY_PATH
)

category_coverage_output_path = write_csv_safely(
    category_coverage_df,
    CATEGORY_COVERAGE_PATH
)

print("Saved final match results:")
print(match_results_output_path)

print("\nSaved match summary:")
print(match_summary_output_path)

print("\nSaved category coverage:")
print(category_coverage_output_path)

print("\nOverall matching summary:")
display(match_summary_df)

print("\nCoverage by taxonomy category:")
display(category_coverage_df)

Saved final match results:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_match_results.csv

Saved match summary:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_match_summary.csv

Saved category coverage:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_opensanctions_category_coverage.csv

Overall matching summary:


,measure,record_count,coverage_rate,definition
0,benchmark_records,675,1.000000,All records in the final Dutch official-source...
1,strict_confirmed_matches,277,0.410370,Unique normalised exact and cleaned exact matc...
2,rule_based_initial_surname_matches,8,0.011852,Unique Dutch-relevant OpenSanctions IDs with m...
3,extended_rule_based_matches,285,0.422222,Strict confirmed matches plus rule-based initi...
4,ambiguous_exact_name_candidates,5,0.007407,Exact-name collisions involving more than one ...
5,unresolved_after_deterministic_matching,385,0.570370,Records not identified by the implemented dete...
6,unresolved_records_with_fuzzy_review_candidates,6,0.008889,Unresolved records with high- or medium-priori...



Coverage by taxonomy category:


,taxonomy_category,benchmark_records,strict_confirmed_matches,rule_based_initial_surname_matches,extended_rule_based_matches,ambiguous_exact_candidates,unresolved_after_deterministic_matching,strict_coverage_rate,extended_coverage_rate
0,house_members,150,146,0,146,4,0,0.973333,0.973333
1,high_judiciary_crvb,86,0,1,1,0,85,0.000000,0.011628
2,ambassadors,82,5,0,5,0,77,0.060976,0.060976
3,registered_party_board,79,9,0,9,1,69,0.113924,0.113924
4,senate_members,75,72,1,73,0,2,0.960000,0.973333
5,high_judiciary_rvs,74,11,0,11,0,63,0.148649,0.148649
6,high_judiciary_cbb,46,0,0,0,0,46,0.000000,0.000000
7,high_judiciary_hr,34,0,6,6,0,28,0.000000,0.176471
8,national_executive,28,28,0,28,0,0,1.000000,1.000000
9,charges_d_affaires,7,0,0,0,0,7,0.000000,0.000000


## 7. Interpretation of matching results

The next cell produces a snapshot-specific interpretation draft directly from the calculated results. This avoids carrying forward figures from an earlier benchmark-development run after the live diplomatic source changed.

Strict confirmed coverage remains the primary estimate. Extended coverage adds the separately reported initial–surname rule-based matches. Fuzzy candidates remain review candidates and are not counted as confirmed coverage.


In [9]:
# -------------------------------------------------------------------
# Create concise tables for thesis reporting and audit
# -------------------------------------------------------------------

THESIS_RESULTS_TABLE_PATH = (
    OUTPUT_DIR / "thesis_table_opensanctions_coverage_summary.csv"
)

THESIS_CATEGORY_TABLE_PATH = (
    OUTPUT_DIR / "thesis_table_opensanctions_category_coverage.csv"
)

RULE_BASED_AUDIT_SAMPLE_PATH = (
    OUTPUT_DIR / "initial_surname_rule_based_audit_sample.csv"
)

# Overall result table formatted for direct use in the thesis.
thesis_results_table_df = match_summary_df[
    match_summary_df["measure"].isin(
        [
            "benchmark_records",
            "strict_confirmed_matches",
            "rule_based_initial_surname_matches",
            "extended_rule_based_matches",
            "ambiguous_exact_name_candidates",
            "unresolved_after_deterministic_matching"
        ]
    )
].copy()

thesis_results_table_df["coverage_percentage"] = (
    thesis_results_table_df["coverage_rate"] * 100
).round(1)

# Category results table with percentage columns suitable for reporting.
thesis_category_table_df = category_coverage_df[
    [
        "taxonomy_category",
        "benchmark_records",
        "strict_confirmed_matches",
        "rule_based_initial_surname_matches",
        "extended_rule_based_matches",
        "unresolved_after_deterministic_matching",
        "strict_coverage_rate",
        "extended_coverage_rate"
    ]
].copy()

thesis_category_table_df["strict_coverage_percentage"] = (
    thesis_category_table_df["strict_coverage_rate"] * 100
).round(1)

thesis_category_table_df["extended_coverage_percentage"] = (
    thesis_category_table_df["extended_coverage_rate"] * 100
).round(1)

thesis_category_table_df = thesis_category_table_df.drop(
    columns=[
        "strict_coverage_rate",
        "extended_coverage_rate"
    ]
)

# A transparent sample of rule-based matches for appendix review.
rule_based_audit_sample_df = (
    final_match_results_df[
        final_match_results_df["match_type"]
        .eq("initial_surname_unique")
    ][
        [
            "benchmark_record_id",
            "person_name",
            "role_title",
            "taxonomy_category",
            "institution",
            "opensanctions_name",
            "opensanctions_id",
            "countries",
            "dataset",
            "match_type",
            "match_confidence",
            "review_status"
        ]
    ]
    .sort_values(
        [
            "taxonomy_category",
            "person_name"
        ]
    )
    .head(20)
    .copy()
)

write_csv_safely(
    thesis_results_table_df,
    THESIS_RESULTS_TABLE_PATH
)

write_csv_safely(
    thesis_category_table_df,
    THESIS_CATEGORY_TABLE_PATH
)

write_csv_safely(
    rule_based_audit_sample_df,
    RULE_BASED_AUDIT_SAMPLE_PATH
)

print("Overall coverage table:")
display(
    thesis_results_table_df[
        [
            "measure",
            "record_count",
            "coverage_percentage",
            "definition"
        ]
    ]
)

print("\nCategory coverage table:")
display(thesis_category_table_df)

print("\nRule-based audit sample:")
display(rule_based_audit_sample_df)

print("\nSaved thesis-ready outputs:")
print(THESIS_RESULTS_TABLE_PATH)
print(THESIS_CATEGORY_TABLE_PATH)
print(RULE_BASED_AUDIT_SAMPLE_PATH)

Overall coverage table:


,measure,record_count,coverage_percentage,definition
0,benchmark_records,675,100.0,All records in the final Dutch official-source...
1,strict_confirmed_matches,277,41.0,Unique normalised exact and cleaned exact matc...
2,rule_based_initial_surname_matches,8,1.2,Unique Dutch-relevant OpenSanctions IDs with m...
3,extended_rule_based_matches,285,42.2,Strict confirmed matches plus rule-based initi...
4,ambiguous_exact_name_candidates,5,0.7,Exact-name collisions involving more than one ...
5,unresolved_after_deterministic_matching,385,57.0,Records not identified by the implemented dete...



Category coverage table:


,taxonomy_category,benchmark_records,strict_confirmed_matches,rule_based_initial_surname_matches,extended_rule_based_matches,unresolved_after_deterministic_matching,strict_coverage_percentage,extended_coverage_percentage
0,house_members,150,146,0,146,0,97.3,97.3
1,high_judiciary_crvb,86,0,1,1,85,0.0,1.2
2,ambassadors,82,5,0,5,77,6.1,6.1
3,registered_party_board,79,9,0,9,69,11.4,11.4
4,senate_members,75,72,1,73,2,96.0,97.3
5,high_judiciary_rvs,74,11,0,11,63,14.9,14.9
6,high_judiciary_cbb,46,0,0,0,46,0.0,0.0
7,high_judiciary_hr,34,0,6,6,28,0.0,17.6
8,national_executive,28,28,0,28,0,100.0,100.0
9,charges_d_affaires,7,0,0,0,7,0.0,0.0



Rule-based audit sample:


,benchmark_record_id,person_name,role_title,taxonomy_category,institution,opensanctions_name,opensanctions_id,countries,dataset,match_type,match_confidence,review_status
201,202,M.M. den Boer,Raadsheer-plaatsvervanger,high_judiciary_crvb,Centrale Raad van Beroep,Monica den Boer,Q2270440,nl,MySociety's EveryPolitician Legislators;PEP po...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
240,241,F.G.F. Peters,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Frederik Peters,Q106119790,nl,Wikidata Persons in Relevant Categories;Wikida...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
245,246,G. de Groot,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Gert de Groot,Q5865178,nl,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
254,255,M.V. Polak,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Marvin Polak,Q106204204,nl,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
255,256,M.W.C. Feteris,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Maarten Feteris,Q17428797,nl,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
257,258,R. Kuiper,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Roel Kuiper,Q3057862,nl,Wikidata Persons in Relevant Categories,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
261,262,T. Kooijmans,"De president, de vice-presidenten, de raadsher...",high_judiciary_hr,hogeraad.nl,Tiny Kooijmans,Q3931118,nl,PEP position annotations by OpenSanctions;Wiki...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit
653,654,R.S. Croll,Lid van de Eerste Kamer der Staten-Generaal,senate_members,Eerste Kamer der Staten-Generaal,Robert Croll,Q117193203,nl,Netherlands Senate;Wikidata Persons in Relevan...,initial_surname_unique,high,rule_based_match_flagged_for_post_hoc_audit



Saved thesis-ready outputs:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\thesis_table_opensanctions_coverage_summary.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\thesis_table_opensanctions_category_coverage.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\initial_surname_rule_based_audit_sample.csv


### Snapshot-specific interpretation draft

This output is generated from the current notebook results. Copy the text into the thesis only after completing the final rerun of notebooks 01–04.


In [10]:
# -------------------------------------------------------------------
# Generate a current-snapshot interpretation draft from calculated data
# -------------------------------------------------------------------

def metric_row(measure_name):
    return match_summary_df.loc[
        match_summary_df["measure"].eq(measure_name)
    ].iloc[0]


def category_row(category_name):
    matching_rows = category_coverage_df.loc[
        category_coverage_df["taxonomy_category"].eq(category_name)
    ]

    if matching_rows.empty:
        return None

    return matching_rows.iloc[0]


benchmark_metric = metric_row("benchmark_records")
strict_metric = metric_row("strict_confirmed_matches")
rule_metric = metric_row("rule_based_initial_surname_matches")
extended_metric = metric_row("extended_rule_based_matches")
unresolved_metric = metric_row("unresolved_after_deterministic_matching")
fuzzy_metric = metric_row(
    "unresolved_records_with_fuzzy_review_candidates"
)

benchmark_total = int(benchmark_metric["record_count"])
strict_count = int(strict_metric["record_count"])
rule_count = int(rule_metric["record_count"])
extended_count = int(extended_metric["record_count"])
unresolved_count = int(unresolved_metric["record_count"])
fuzzy_count = int(fuzzy_metric["record_count"])

strict_pct = strict_metric["coverage_rate"] * 100
rule_pct = rule_metric["coverage_rate"] * 100
extended_pct = extended_metric["coverage_rate"] * 100
unresolved_pct = unresolved_metric["coverage_rate"] * 100
fuzzy_upper_bound_pct = (
    (extended_count + fuzzy_count)
    / benchmark_total
    * 100
)

senate_row = category_row("senate_members")

senate_text = ""
if senate_row is not None:
    senate_text = (
        f" For Senate members, {int(senate_row['strict_confirmed_matches'])} "
        f"of {int(senate_row['benchmark_records'])} records "
        f"({senate_row['strict_coverage_rate'] * 100:.1f}%) were strictly "
        f"matched, increasing to "
        f"{int(senate_row['extended_rule_based_matches'])} "
        f"({senate_row['extended_coverage_rate'] * 100:.1f}%) after the "
        "initial–surname rule."
    )

interpretation_text = (
    "The benchmark used in this analysis is the current documented "
    f"official-source snapshot and contains {benchmark_total} records. "
    "Using strict confirmed matching only, the procedure identified "
    f"{strict_count} records ({strict_pct:.1f}%) in the OpenSanctions "
    "export. A secondary sensitivity analysis added "
    f"{rule_count} unique initial–surname matches ({rule_pct:.1f}% of "
    "the benchmark), producing an extended rule-based estimate of "
    f"{extended_count} records ({extended_pct:.1f}%)."
    + senate_text +
    f" A total of {unresolved_count} records ({unresolved_pct:.1f}%) "
    "were not identified by the implemented deterministic matching "
    "procedure. Fuzzy candidates were not counted as matches: "
    f"{fuzzy_count} unresolved records had high- or medium-priority "
    "candidates. If all of these were manually confirmed, the extended "
    f"estimate would rise to at most {fuzzy_upper_bound_pct:.1f}%; this "
    "is an exploratory upper bound rather than a confirmed coverage "
    "estimate. Unmatched records may reflect name-format differences, "
    "aliases, source timing, incomplete country metadata, or genuine "
    "absence from the OpenSanctions export."
)

INTERPRETATION_DRAFT_PATH = (
    OUTPUT_DIR
    / "thesis_interpretation_current_snapshot.txt"
)

INTERPRETATION_DRAFT_PATH.write_text(
    interpretation_text,
    encoding="utf-8"
)

print(interpretation_text)
print("\nSaved interpretation draft:")
print(INTERPRETATION_DRAFT_PATH)


The benchmark used in this analysis is the current documented official-source snapshot and contains 675 records. Using strict confirmed matching only, the procedure identified 277 records (41.0%) in the OpenSanctions export. A secondary sensitivity analysis added 8 unique initial–surname matches (1.2% of the benchmark), producing an extended rule-based estimate of 285 records (42.2%). For Senate members, 72 of 75 records (96.0%) were strictly matched, increasing to 73 (97.3%) after the initial–surname rule. A total of 385 records (57.0%) were not identified by the implemented deterministic matching procedure. Fuzzy candidates were not counted as matches: 6 unresolved records had high- or medium-priority candidates. If all of these were manually confirmed, the extended estimate would rise to at most 43.1%; this is an exploratory upper bound rather than a confirmed coverage estimate. Unmatched records may reflect name-format differences, aliases, source timing, incomplete country metadat

# Findings 

The benchmark used in this analysis is the current documented official-source snapshot and contains 675 records. The primary coverage estimate is based on strict confirmed matches: unique normalised exact matches and cleaned exact matches. Under this conservative definition, the implemented procedure identified 277 benchmark records in the OpenSanctions export, corresponding to strict coverage of 41.0%.

A secondary sensitivity analysis added eight unique initial--surname matches. These records had a matching full surname component, matching first initial, and one unique candidate OpenSanctions identifier under the implemented rule. Including these abbreviation-aware records produced an extended rule-based coverage estimate of 285 of 675 benchmark records, or 42.2%. These rule-based matches were retained separately from strict confirmed matches and flagged for post-hoc audit.

The enrichment of Eerste Kamer records illustrates the importance of source-specific name treatment. The official Senate member-list page often presents members using initials, titles, or degree suffixes, whereas linked official profile pages usually provide the full name. The benchmark retained all 75 current Senate members and obtained profile-derived full names for 73 records. As a result, 72 of 75 Senate records (96.0%) were strictly matched in OpenSanctions. Including one additional initial--surname match increased Senate coverage to 73 of 75 records (97.3%). This indicates that apparent coverage gaps can partly arise from differences in the presentation of official names rather than absent OpenSanctions records.

A total of 385 benchmark records (57.0%) were not identified by the implemented deterministic matching procedure. These records should not automatically be interpreted as absent from OpenSanctions. Potential explanations include unresolved name-format differences, aliases not present in the export, source-timing differences, incomplete country metadata, or genuine absence from the OpenSanctions export.

Fuzzy candidates were not counted as confirmed matches. Six unresolved benchmark records had high- or medium-priority fuzzy candidates, but string similarity alone does not establish identity. If all six candidates were later manually confirmed, the extended coverage estimate would increase from 42.2% to a maximum of 43.1%. This figure is therefore treated as an exploratory upper bound rather than a confirmed coverage estimate.
